# LangChain Revision Assignment — 8th October 2026

This assignment covers every topic in the `langchain/` folder:

| # | Section | Source folder |
|---|---------|---------------|
| 1 | Models — LLMs vs Chat Models | `models/LLMs`, `models/ChatModels` |
| 2 | Embedding Models | `models/EmbededModels` |
| 3 | Prompts — PromptTemplate, Messages, ChatPromptTemplate | `prompts/` |
| 4 | Structured Output — TypedDict, Pydantic, JSON Schema | `output/structuredoutput` |
| 5 | Output Parsers — Str, Json, Structured, Pydantic | `output/outputParsers` |
| 6 | Chains — simple, sequential, parallel, conditional | `chains/` |
| 7 | Runnables — Sequence, Parallel, Passthrough, Lambda, Branch | `runnables/` |
| 8 | Document Loaders — Text, PDF, Directory, CSV, Web | `docloaders/` |
| 9 | Text Splitters — Character, Recursive, Language, Semantic | `Splitters/` |
| 10 | Retrievers *(bonus)* | `retrievers/` |
| 11 | Tools — built-in, custom, toolkits | `Tools/` |
| 12 | Tool Calling — bind_tools, ToolMessage, currency conversion | `Tool Calling/` |
| 13 | Agents — ReAct and tool-calling agents | `endToendProject/` |
| 14 | Capstone project | everything |

**How to work through it**
- Each section opens with **concept questions**. Write your answers in the markdown cell under them, without looking at your notes.
- Then complete the **coding tasks**. Replace every `# TODO` and every `...`.
- Cells marked ✅ contain `assert`s that check your work. A task is done once they pass.
- Rough marks are given per section: **200** core marks, plus a 10-mark bonus and a 20-mark capstone. Try to finish in one sitting (~3–4 hrs).

## 0. Setup
Make sure your `.env` (at the repo root) has `OPENAI_API_KEY` (and optionally `HUGGINGFACEHUB_API_TOKEN`).

In [ ]:
# !pip install langchain langchain-core langchain-community langchain-openai langchain-huggingface \
#              langchain-text-splitters langchain-experimental langchain-classic pypdf ddgs scikit-learn python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Paths to the resources used in your notes
RESOURCES = Path("../../langchain/docloaders/resources").resolve()
PDF_PATH = RESOURCES / "dl-curriculum.pdf"
TXT_PATH = RESOURCES / "cricket.txt"

assert PDF_PATH.exists() and TXT_PATH.exists(), "Fix the RESOURCES path"
print("OPENAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))

---
## 1. Models — LLMs vs Chat Models  *(10 marks)*

### Concept questions
1. What is the difference between `OpenAI` (LLM) and `ChatOpenAI` (chat model), considering both the input and the return type?
2. Why has LangChain moved away from plain LLMs and focused on chat models?
3. Explain `temperature`. What value would you use for (a) a SQL generator, (b) a poem writer?
4. What does `max_completion_tokens` control?
5. Name 3 useful fields inside `response.response_metadata` / `response.usage_metadata`.

**Your answers:**
A

### Task 1.1
Create a `ChatOpenAI` model with `temperature=0` and `max_completion_tokens=100`. Ask it *"What is the capital of India?"*. Print:
- the `.content`
- the type of the response object
- the total tokens used (from `usage_metadata`)

In [6]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model = 'gpt-4', temperature = 0.0, max_tokens = 1000)
response = chatModel.invoke("What is the captial of India")

print(response.content)
print(response.response_metadata['token_usage']['total_tokens'])


# TODO: print content, type(response), total tokens

The capital of India is New Delhi.
22


### Task 1.2 — Temperature experiment
Invoke the **same** prompt *"Give me a one-line name idea for a cricket academy"* 3 times with `temperature=0` and 3 times with `temperature=1.5`. Store the outputs in two lists and print them. In a comment, write what you observe.

In [ ]:
prompt_text = "Give me a one-line name idea for a cricket academy"

low_temp_outputs = []   # TODO
high_temp_outputs = []  # TODO


# Observation:

---
## 2. Embedding Models  *(12 marks)*

### Concept questions
1. What is an embedding? Why is it useful?
2. Difference between `embed_query` and `embed_documents`?
3. What does the `dimensions` parameter in `OpenAIEmbeddings(model="text-embedding-3-small", dimensions=32)` do? What is the trade-off of a smaller dimension?
4. Why do we use **cosine similarity** rather than Euclidean distance for comparing text embeddings?
5. Name one open-source embedding model you could use with `HuggingFaceEmbeddings`.

**Your answers:**

### Task 2.1
Create an `OpenAIEmbeddings` model with `dimensions=64`. Embed one query and a list of 3 documents. ✅ Verify the shapes.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = ...  # TODO
q_vec = ...            # TODO: embed "What is machine learning?"
doc_vecs = ...         # TODO: embed 3 sentences of your choice

In [19]:
# ✅ checks
assert len(q_vec) == 64
assert len(doc_vecs) == 3 and all(len(v) == 64 for v in doc_vecs)
print("Task 2.1 passed")

NameError: name 'q_vec' is not defined

### Task 2.2 — Mini semantic search engine
Write a function `semantic_search(query, docs, k=2)` that returns the **top-k** most similar documents along with their scores, sorted by score in descending order. Use `cosine_similarity` from sklearn.

Test it with the cricket sentences below and the query *"Who is the best captain?"*.

In [17]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from langchain_openai import OpenAIEmbeddings

docs = [
    "Kohli is the best batsman in the world",
    "Rohit Sharma is a good batsman",
    "Dhoni is a great captain",
    "Sachin is the god of cricket",
    "Bumrah is the best fast bowler",
]

def semantic_search(query: str, docs: list[str], k: int = 2) -> list[tuple[str, float]]:
    embeddingModel = OpenAIEmbeddings()
    documents_vector = embeddingModel.embed_documents(docs)
    query_vector = embeddingModel.embed_query(query)
    answers: list[tuple[str, float]] = []
    for(doc, doc_vector) in zip(docs, documents_vector):
        similarity = cosine_similarity([query_vector], [doc_vector])
        print(f"Similarity between '{query}' and '{doc}': {similarity[0][0]}")
        # answers.append((doc, similarity[0][0]))
    # print(answers)
    return answers.sort(key=lambda x: x[1], reverse=True)[:k]

    ...

def semantic_search(query: str, docs: list[str], k: int = 2) -> list[tuple[str, float]]:
    embedding_model = OpenAIEmbeddings()
    doc_vectors = embedding_model.embed_documents(docs)
    query_vector = embedding_model.embed_query(query)

    scored = []
    for doc, doc_vector in zip(docs, doc_vectors):
        score = cosine_similarity([query_vector], [doc_vector])[0][0]
        scored.append((doc, float(score)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]

results = semantic_search("Who is the best captain?", docs, k=2)
print(results)

[('Dhoni is a great captain', 0.8596548615527575), ('Kohli is the best batsman in the world', 0.8120347483008217)]


In [18]:
# ✅ checks
assert len(results) == 2
assert results[0][1] >= results[1][1], "results must be sorted by score"
assert "Dhoni" in results[0][0]
print("Task 2.2 passed")

Task 2.2 passed


### Task 2.3 *(optional, needs HF)*
Repeat Task 2.1 with `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")`. What is the embedding dimension? Can you change it?

---
## 3. Prompts  *(20 marks)*

### Concept questions
1. Why use `PromptTemplate` instead of an f-string?
2. What does `validate_template=True` do?
3. How do you **save** a prompt template to disk and **load** it back? Why is this useful?
4. Name the 3 message types and what each represents.
5. Your notes say that inside a `ChatPromptTemplate`, `SystemMessage(content="You are a {domain} expert")` does **not** substitute `{domain}`. Why not, and what format should you use instead?
6. What is `MessagesPlaceholder` used for?
7. What is the difference between a **static** prompt and a **dynamic** prompt?

**Your answers:**

### Task 3.1 — PromptTemplate + save/load
1. Create a `PromptTemplate` that explains a research paper given `paper_input`, `style_input`, `length_input`. Use `validate_template=True`.
2. Save it to `paper_template.json` in this folder.
3. Load it back using `load_prompt` and invoke it with sample values. Print the final prompt string.

In [ ]:
from langchain_core.prompts import PromptTemplate, load_prompt

template = ...  # TODO

# TODO save
# TODO load
loaded = ...

filled = ...  # TODO invoke loaded template
print(filled.text)

In [ ]:
# ✅ checks
assert set(loaded.input_variables) == {"paper_input", "style_input", "length_input"}
print("Task 3.1 passed")

### Task 3.2 — Messages
Build a list `[SystemMessage, HumanMessage]`, invoke the chat model, then **append** the reply as an `AIMessage`. Ask a follow-up question that depends on the first answer (e.g. *"What is its population?"*) and show that the model remembers the context.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    # TODO
]

# TODO: first invoke, append AIMessage, append follow-up HumanMessage, invoke again

### Task 3.3 — Console chatbot with history
Write a small `while True` chatbot loop that:
- keeps `chat_history` as a list of messages (starting with a SystemMessage)
- breaks when the user types `exit`
- prints the AI response each turn

*(Tip: test with 3–4 turns, then type `exit`.)*

In [ ]:
chat_history = [SystemMessage(content="You are a helpful assistant")]

# TODO

### Task 3.4 — ChatPromptTemplate + MessagesPlaceholder
Create a `ChatPromptTemplate` with:
- a system message: `"You are a {domain} expert"`
- a `MessagesPlaceholder` named `chat_history`
- a human message: `"{query}"`

Invoke it with `domain="customer support"`, a 2-message history, and `query="Where is my refund?"`. Print the resulting messages. ✅ There should be 4 messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_template = ...  # TODO

history = [
    HumanMessage(content="I want a refund for order #1234"),
    AIMessage(content="Your refund has been initiated."),
]

final_prompt = ...  # TODO
print(final_prompt)

In [ ]:
# ✅ checks
msgs = final_prompt.to_messages()
assert len(msgs) == 4
assert "customer support" in msgs[0].content
print("Task 3.4 passed")

### Task 3.5 *(optional)* — Streamlit UI
Re-create `prompt_template_ui.py` from memory in a separate file `practice/8th_october_test/app.py`: 3 dropdowns plus a submit button that runs `template | llm`. Run it with `streamlit run app.py`.

---
## 4. Structured Output with `with_structured_output`  *(18 marks)*

### Concept questions
1. What does `model.with_structured_output(schema)` return?
2. List the 3 ways to define a schema. When would you pick each one?
3. What do `Annotated`, `Optional`, and `Literal` do inside a `TypedDict`?
4. Does a `TypedDict` schema **validate** the data? Does Pydantic?
5. How do you convert a Pydantic object to a dict and to a JSON string (Pydantic v2)?
6. Some models (e.g. many open-source ones) don't support structured output natively. What do you use then?

**Your answers:**

Use this review for all tasks in this section:

In [ ]:
review_text = """
I bought the boAt Airdopes 141 last month. The sound quality is surprisingly good for the price and the
bass is punchy. Battery easily lasts 2 days. However, the call quality is poor in noisy places and the
touch controls are too sensitive — I keep pausing music by accident. For ₹1,299 it's still a good deal.
Reviewed by Priya Sharma
"""

### Task 4.1 — TypedDict
Define a `ProductReview` `TypedDict` with: `summary` (str), `sentiment` (Literal `"pos"`/`"neg"`/`"neutral"`), `pros` (Optional list), `cons` (Optional list), `price` (Optional str), and `reviewer` (Optional str). Use `Annotated` descriptions. Invoke the structured model and print `result["cons"]`.

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal

class ProductReview(TypedDict):
    # TODO
    ...

structured_llm = ...  # TODO
result_td = ...       # TODO
print(result_td)

### Task 4.2 — Pydantic
Re-do 4.1 with a Pydantic `BaseModel`. Add a field `rating: int` with a constraint of `ge=1, le=5` (the model has to infer it). Print the object, then `.model_dump()` and `.model_dump_json()`.

In [ ]:
from pydantic import BaseModel, Field

class ProductReviewModel(BaseModel):
    # TODO
    ...

result_pd = ...  # TODO

In [ ]:
# ✅ checks
assert isinstance(result_pd, ProductReviewModel)
assert 1 <= result_pd.rating <= 5
print("Task 4.2 passed")

### Task 4.3 — JSON Schema
Write the same schema as a raw JSON-schema `dict`, with `required` set to `["summary", "sentiment"]`. What type does the result come back as?

In [ ]:
review_json_schema = {
    # TODO
}

result_js = ...  # TODO
print(type(result_js), result_js)

---
## 5. Output Parsers  *(18 marks)*

### Concept questions
1. Why do we need `StrOutputParser` when chaining two LLM calls?
2. What is `parser.get_format_instructions()`, and why is it passed as a **partial variable**?
3. What is the limitation of `JsonOutputParser`?
4. What does `StructuredOutputParser` add over `JsonOutputParser`, and what is it still missing?
5. When would you use `PydanticOutputParser` instead of `with_structured_output`?

**Your answers:**

### Task 5.1 — StrOutputParser chain
Build a chain: *detailed report on `{topic}`* → model → parser → *5-line summary of `{text}`* → model → parser. Invoke it with `topic="black holes"`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# TODO

### Task 5.2 — JsonOutputParser
Ask for *5 facts about `{topic}`* using `JsonOutputParser`, with format instructions as a partial variable. Print the result's `type`.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

# TODO

### Task 5.3 — StructuredOutputParser
Use `ResponseSchema` to get exactly `fact_1`, `fact_2`, `fact_3` about a topic.
*(Hint: in newer versions it lives in `langchain_classic.output_parsers` or `langchain.output_parsers`.)*

In [ ]:
# TODO

### Task 5.4 — PydanticOutputParser
Define a `Person` model (`name: str`, `age: int = Field(gt=18)`, `city: str`). Build `template | model | parser` to generate a fictional person from `{place}`.

**Then:** call `parser.parse('{"name": "Ravi", "age": 10, "city": "Pune"}')` directly inside a try/except, and print the validation error. Explain why this happens.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

# TODO

---
## 6. Chains  *(16 marks)*

### Concept questions
1. What does the `|` operator do between two runnables?
2. How do you visualise a chain?
3. In a parallel chain, how does the output of each branch reach the next prompt?
4. In a conditional chain, what does the **last** (non-tuple) argument to `RunnableBranch` represent?

**Your answers:**

### Task 6.1 — Simple chain + graph
Build `prompt | model | parser` that generates 5 interesting facts about `{topic}`. Invoke it and print the ASCII graph.

In [ ]:
# TODO

### Task 6.2 — Parallel chain
Given a block of text, **in parallel** generate (a) short notes and (b) 5 quiz questions, then **merge** them into a single document with a third prompt. Print the graph to confirm the parallel structure.

In [ ]:
from langchain_core.runnables import RunnableParallel

sample_text = """
Support Vector Machines (SVMs) are supervised learning methods used for classification, regression and outlier
detection. They are effective in high dimensional spaces and use a subset of training points (support vectors)
in the decision function, making them memory efficient. Different kernel functions can be specified.
"""

# TODO

### Task 6.3 — Conditional chain (feedback router)
1. Classify a customer review's sentiment into `positive`/`negative` using `PydanticOutputParser`.
2. Use `RunnableBranch` to:
   - **positive** → generate a thank-you reply
   - **negative** → generate an apology reply
   - default → return `"Could not determine sentiment"`
3. Test it with one positive and one negative review.

*Bonus:* Make the reply prompts **dynamic** so they include the original review text. (Hint: you need the original input as well as the sentiment. Think about `RunnableParallel` / `RunnablePassthrough`.)

In [ ]:
from typing import Literal
from langchain_core.runnables import RunnableBranch, RunnableLambda

# TODO

---
## 7. Runnables  *(18 marks)*

### Concept questions
1. What problem did Runnables solve? (Think about the old `LLMChain` and friends, and the common `invoke` interface.)
2. Explain each of the following in one line: `RunnableSequence`, `RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`, `RunnableBranch`.
3. What is the input and output type of `RunnableParallel`?
4. Is `prompt | model | parser` the same as `RunnableSequence(prompt, model, parser)`?
5. Name 3 methods every Runnable has besides `invoke`.

**Your answers:**

### Task 7.1 — No-LLM warm-up ✅
Without any model, using only `RunnableLambda`, `RunnablePassthrough`, `RunnableParallel`, `RunnableSequence`, and `RunnableBranch`, build `pipeline` such that:
- input: a string
- step 1: strip and lowercase it
- step 2: in parallel produce `{"text": <the string>, "words": <word count>, "chars": <char count>}`
- step 3: branch. If `words > 5`, add `"label": "long"`, otherwise add `"label": "short"`.

In [ ]:
from langchain_core.runnables import (
    RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch
)

pipeline = ...  # TODO

In [ ]:
# ✅ checks
out1 = pipeline.invoke("  Hello World  ")
assert out1 == {"text": "hello world", "words": 2, "chars": 11, "label": "short"}, out1
out2 = pipeline.invoke("LangChain runnables make composing pipelines really easy")
assert out2["label"] == "long"
print(pipeline.batch(["a b c", "one two three four five six"]))
print("Task 7.1 passed")

### Task 7.2 — Passthrough
Generate a joke about `{topic}`, then in parallel return the **joke itself** (passthrough) and an **explanation** of the joke.

In [ ]:
# TODO

### Task 7.3 — Lambda
Generate a joke and return `{"joke": ..., "word_count": ...}` using `RunnableLambda`.

In [ ]:
# TODO

### Task 7.4 — Branch
Write a detailed report on `{topic}`. If it has more than 300 words, summarise it; otherwise pass it through unchanged.

In [ ]:
# TODO

### Task 7.5 — Streaming & batch
Using the chain from 6.1:
- `.stream()` the output and print tokens as they arrive
- `.batch()` it over 3 topics

In [ ]:
# TODO

---
## 8. Document Loaders  *(16 marks)*

### Concept questions
1. What does every loader return? What are the two attributes of a `Document`?
2. Difference between `load()` and `lazy_load()`? When must you use `lazy_load()`?
3. `PyPDFLoader` creates one `Document` per ______.
4. What does `glob` and `loader_cls` do in `DirectoryLoader`?
5. What metadata does `WebBaseLoader` add? Can it load multiple URLs?
6. In `CSVLoader`, what does one Document correspond to?

**Your answers:**

### Task 8.1 — TextLoader
Load `cricket.txt`. Print the number of documents, the metadata, and the first 200 characters. Then build a chain that **summarises** the text into 3 bullet points.

In [ ]:
from langchain_community.document_loaders import TextLoader

# TODO

### Task 8.2 — PyPDFLoader ✅
Load `dl-curriculum.pdf`. Print the page count and the metadata of page 0.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_docs = ...  # TODO

In [ ]:
# ✅ checks
assert len(pdf_docs) > 1
assert "page" in pdf_docs[0].metadata
print("Task 8.2 passed")

### Task 8.3 — DirectoryLoader with lazy_load
Load every PDF in `RESOURCES` with `DirectoryLoader(..., glob="**/*.pdf", loader_cls=PyPDFLoader)`. Use `lazy_load()` and count the pages **without** keeping them all in memory.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# TODO

### Task 8.4 — CSVLoader
The cell below creates a small CSV. Load it with `CSVLoader`. ✅ Check that there is one Document per row, and print `docs[1].page_content`.

In [ ]:
import csv
from langchain_community.document_loaders import CSVLoader

csv_path = Path("players.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["name", "role", "runs"])
    w.writerows([["Kohli", "Batsman", 13848], ["Bumrah", "Bowler", 120], ["Jadeja", "All-rounder", 2756]])

csv_docs = ...  # TODO

In [ ]:
# ✅ checks
assert len(csv_docs) == 3
assert "Bumrah" in csv_docs[1].page_content
print("Task 8.4 passed")

### Task 8.5 — WebBaseLoader Q&A
Load any web page (e.g. a Wikipedia article) and build a chain that answers a `{question}` from the page `{text}`.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# TODO

---
## 9. Text Splitters  *(18 marks)*

### Concept questions
1. Why do we split documents before embedding them? Give 3 reasons.
2. What do `chunk_size` and `chunk_overlap` mean? Why is overlap useful?
3. How does `RecursiveCharacterTextSplitter` decide where to split? What is its default separator order?
4. Why would you use `RecursiveCharacterTextSplitter.from_language(Language.PYTHON, ...)` for code?
5. How does `SemanticChunker` decide where to break? What do `breakpoint_threshold_type` and `breakpoint_threshold_amount` mean?
6. `split_text` vs `split_documents` vs `create_documents`?

**Your answers:**

### Task 9.1 — Character vs Recursive ✅
Split the same text with `CharacterTextSplitter(chunk_size=100, chunk_overlap=0, separator="")` and with `RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)`. Print both sets of chunks. In a comment, explain which one keeps sentences/words intact and why.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

text = """LangChain is a framework for building LLM applications. It provides loaders, splitters, retrievers and agents.

Splitting is important because models have context limits. Good chunks keep related sentences together.
Overlap helps preserve context across chunk boundaries."""

char_chunks = ...  # TODO
rec_chunks = ...   # TODO

In [ ]:
# ✅ checks
assert all(len(c) <= 100 for c in char_chunks + rec_chunks)
assert not any(c.endswith(" ") or c.startswith(" ") for c in rec_chunks)
print("Task 9.1 passed")

### Task 9.2 — Overlap
Split the PDF documents with `chunk_size=300, chunk_overlap=50` using `split_documents`. Print the number of chunks, and show the overlapping region between chunk 0 and chunk 1. Does the metadata survive splitting?

In [ ]:
# TODO

### Task 9.3 — Language-aware splitting
Split the Python code below with `from_language(Language.PYTHON, chunk_size=150, chunk_overlap=0)`. ✅ Each `class`/`def` should start its own chunk where possible.

In [ ]:
from langchain_text_splitters import Language

py_code = """
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    def is_passing(self):
        return self.grade >= 6.0


def top_students(students):
    return [s for s in students if s.grade > 9]


if __name__ == "__main__":
    print(top_students([Student("Aarav", 9.5)]))
"""

code_chunks = ...  # TODO
for i, c in enumerate(code_chunks):
    print(f"--- chunk {i} ---\n{c}")

### Task 9.4 — SemanticChunker
Use `SemanticChunker(OpenAIEmbeddings(), breakpoint_threshold_type="standard_deviation", breakpoint_threshold_amount=1)` on the mixed-topic text below. How many chunks do you get? Try `"percentile"` as the threshold type and compare.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker  # or langchain_text_splitters in newer versions

mixed = """
Farmers were working hard in the fields, preparing the soil and planting seeds for the next season. The sun was bright.
The Indian Premier League (IPL) is the biggest cricket league in the world. Fans cheer for their favourite teams.
Terrorism is a big danger to peace and safety. To fight it, we need strong laws and alert security forces.
"""

# TODO

---
## 10. Retrievers *(bonus, 10 marks)*

Your `retrievers/` folder is still empty, so this section previews it. It joins **loaders → splitters → embeddings** together.

### Concept questions
1. What is a retriever? How is it different from a vector store?
2. Is a retriever a Runnable? What does `retriever.invoke(query)` return?
3. What is MMR (Maximal Marginal Relevance), and why would you use it?

**Your answers:**

### Task 10.1
1. Take the PDF chunks from Task 9.2.
2. Put them in an `InMemoryVectorStore` (from `langchain_core.vectorstores`) using `OpenAIEmbeddings`.
3. Create `retriever = vector_store.as_retriever(search_kwargs={"k": 3})`.
4. Query: *"Which topics are covered under CNNs?"*, and print the retrieved chunks with their page numbers.
5. Try `search_type="mmr"` and compare the results.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

# TODO

### Task 10.2 — Mini RAG chain
Build: `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()`.
The prompt must tell the model to answer **only** from the context, and to say "I don't know" otherwise. Ask one question that is answerable and one that is not.

In [ ]:
# TODO

---
## 11. Tools  *(16 marks)*

### Concept questions
1. What is a tool in LangChain? Which 3 attributes does the LLM see?
2. Why are **type hints** and a **docstring** mandatory when using `@tool`?
3. Compare `@tool`, `StructuredTool.from_function`, and subclassing `BaseTool`. When would you use each?
4. What is a toolkit?
5. Why is `ShellTool` dangerous to give to an LLM?

**Your answers:**

### Task 11.1 — Built-in tool
Use `DuckDuckGoSearchRun` to search for *"latest IPL winner"* and pipe it into a `StrOutputParser`.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

# TODO

### Task 11.2 — `@tool` decorator ✅
Create a `@tool` named `celsius_to_fahrenheit(celsius: float) -> float`. Print its `name`, `description`, `args`, and its `args_schema.model_json_schema()`.

In [ ]:
from langchain_core.tools import tool

# TODO

In [ ]:
# ✅ checks
assert celsius_to_fahrenheit.invoke({"celsius": 100}) == 212
assert "celsius" in celsius_to_fahrenheit.args
print("Task 11.2 passed")

### Task 11.3 — StructuredTool ✅
Create a function `bmi(weight_kg, height_m)` and wrap it with `StructuredTool.from_function`. Give it a Pydantic `args_schema` with field descriptions and the constraint `gt=0`, plus a custom name `BMICalculator`.

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

# TODO
bmi_tool = ...

In [ ]:
# ✅ checks
assert bmi_tool.name == "BMICalculator"
assert round(bmi_tool.invoke({"weight_kg": 70, "height_m": 1.75}), 1) == 22.9
try:
    bmi_tool.invoke({"weight_kg": -5, "height_m": 1.75})
    raise AssertionError("should have failed validation")
except Exception as e:
    print("validation works:", type(e).__name__)
print("Task 11.3 passed")

### Task 11.4 — BaseTool subclass ✅
Create `WordCountTool(BaseTool)` with an `args_schema` and a `_run(self, text: str) -> int`.

In [ ]:
from langchain_core.tools import BaseTool
from typing import Type

# TODO
word_tool = ...

In [ ]:
# ✅ checks
assert word_tool.invoke({"text": "one two three"}) == 3
print("Task 11.4 passed")

### Task 11.5 — Toolkit ✅
Create a `UnitToolkit` class with `get_tools()` that returns the 3 tools above. Loop over them and print `name => description`.

In [ ]:
# TODO

---
## 12. Tool Calling  *(18 marks)*

### Concept questions
1. What does `llm.bind_tools([...])` do? Does the LLM **execute** the tool?
2. What is inside `ai_message.tool_calls`? List the keys.
3. What happens when you call `tool.invoke(tool_call)` with the **whole tool-call dict** instead of just the args? What type comes back?
4. Why must you send the `ToolMessage` back to the LLM? What links it to the original call?
5. What is `InjectedToolArg`, and why would you hide an argument from the LLM?

**Your answers:**

### Task 12.1 — Manual tool-calling loop
Using your `multiply` tool:
1. `llm_with_tools.invoke("Hi how are you")` — confirm `tool_calls` is empty.
2. Ask `"multiply 7 by 6"`, grab `tool_calls[0]`, invoke the tool with it, append the `ToolMessage`, and get the final answer.
3. Print the full `messages` list and identify each message type.

In [ ]:
from langchain_core.messages import ToolMessage

@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

# TODO

### Task 12.2 — Generic tool executor
Write `run_with_tools(llm, tools, user_query)`. It should keep calling the LLM and executing **every** tool call (there may be several in one response) until the LLM returns a response with no tool calls, then return the final content.

Test: *"What is 12 * 8, and what is 100 Celsius in Fahrenheit?"* with `[multiply, celsius_to_fahrenheit]`.

In [ ]:
def run_with_tools(llm, tools, user_query: str) -> str:
    tools_by_name = {t.name: t for t in tools}
    # TODO
    ...

print(run_with_tools(ChatOpenAI(), [multiply, celsius_to_fahrenheit], "What is 12 * 8, and what is 100 Celsius in Fahrenheit?"))

### Task 12.3 — Currency Conversion (the `2.CurrentConversion` notebook)
Build two tools:
- `get_conversion_factor(base_currency: str, target_currency: str) -> float`. Hard-code a small rates dict (or call a free API such as `https://open.er-api.com/v6/latest/{base}`).
- `convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float`

Flow: ask *"Convert 10 USD to INR"*. The LLM will call both tools. **You** must inject the `conversion_rate` returned by the first tool into the second call's args before invoking it.

In [ ]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import json

# TODO

---
## 13. Agents  *(20 marks)*

### Concept questions
1. Chain vs agent: what is the key difference?
2. Explain the ReAct loop (Thought → Action → Action Input → Observation → … → Final Answer).
3. What is the role of `AgentExecutor`, and of the `agent_scratchpad`?
4. Why does a **multi-argument** tool (e.g. `calculateOld(a, b, op)`) often fail with `create_react_agent` but work with `create_tool_calling_agent`?
5. In the tool-calling agent prompt, why must `agent_scratchpad` be a `MessagesPlaceholder` and not a string slot?
6. Name 2 useful `AgentExecutor` parameters besides `agent`, `tools`, and `verbose`. (Hint: think about loops and bad output.)

**Your answers:**

### Task 13.1 — ReAct agent
Pull `hwchase17/react` from the hub. Create a ReAct agent with `DuckDuckGoSearchResults` and a **single-string-input** `calculate(expression: str)` tool, which must validate its input with a regex before evaluating it. Ask: *"What is the population of Karnataka plus the population of Kerala?"* and read the verbose trace.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_classic.agents import AgentExecutor, create_react_agent
from langsmith import Client
import re

# TODO

### Task 13.2 — Tool-calling agent
Rebuild the agent with `create_tool_calling_agent` and a `ChatPromptTemplate` (system + human + `MessagesPlaceholder("agent_scratchpad")`). Use a **multi-arg** calculator tool. Set `max_iterations=5` and `handle_parsing_errors=True`. Compare the traces with 13.1. Which one is cleaner?

In [ ]:
from langchain_classic.agents import create_tool_calling_agent

# TODO

### Task 13.3 — Trip Planner agent
Add a `get_weather(city: str)` tool (you can fake it or use the free `https://wttr.in/{city}?format=3` API) and a `budget_split(total: float, days: int)` tool. Ask:
> *"Plan a 3-day trip to Goa with a ₹30,000 budget. Check the weather and split the budget per day."*

Make sure the trace shows all three tools being used.

In [ ]:
# TODO

---
## 14. Capstone — "Study Buddy" over the DL curriculum PDF  *(20 marks)*

Build a single pipeline that uses at least **one concept from every section**:

1. **Load** `dl-curriculum.pdf` (loaders) and **split** it (splitters).
2. **Embed** it and expose a **retriever** (embeddings + retrievers).
3. Wrap the retriever as a **tool** `search_curriculum(query: str)` (tools).
4. Create a `generate_quiz` chain that returns a **Pydantic** `Quiz` object containing a list of `Question(question, options: list[str], answer)` (structured output).
5. Use **RunnableParallel** to produce `{"summary": ..., "quiz": ...}` for a given topic (runnables/chains).
6. Use **RunnableBranch** so that if the topic isn't found in the curriculum, it returns `"Topic not in curriculum"` (conditional chain).
7. Finally, expose it all through a **tool-calling agent** with a `ChatPromptTemplate` + chat history (`MessagesPlaceholder`), so you can have a multi-turn conversation like:
   - *"What does the curriculum say about RNNs?"*
   - *"Make me a 3-question quiz on that."*

**Deliverable:** run a 3-turn conversation and print the outputs.

In [ ]:
# TODO: Capstone

---
## Self-evaluation

| Section | Marks | Score |
|---|---|---|
| 1. Models | 10 | |
| 2. Embeddings | 12 | |
| 3. Prompts | 20 | |
| 4. Structured Output | 18 | |
| 5. Output Parsers | 18 | |
| 6. Chains | 16 | |
| 7. Runnables | 18 | |
| 8. Document Loaders | 16 | |
| 9. Text Splitters | 18 | |
| 10. Retrievers (bonus) | 10 | |
| 11. Tools | 16 | |
| 12. Tool Calling | 18 | |
| 13. Agents | 20 | |
| 14. Capstone | 20 | |
| **Total** | **230 (200 + 10 bonus + 20 capstone)** | |

**Topics I need to revise again:**